In [ ]:
STEPS:
- scrape data using yt-dlp (use last cell to build processed_videos.txt if not updated) (rotating proxies to evade ip bans)
- cell just below loads data
- 


In [ ]:
import os
import json
import glob
import re
import time
from difflib import SequenceMatcher
from datetime import datetime, timedelta
import pandas as pd
from google import genai

# =========================
# 🔑 CONFIG
# =========================
# Ensure your API Key is handled securely in production environment
client = genai.Client(api_key="api key")

BATCH_SIZE = 8

# =========================
# 🧠 UPDATED PREPROMPT (Aligned with Project Scope)
# =========================
# This prompt enforces the Taxonomy Reference  and Data Schema 
PREPROMPT = """
You are an expert insurance analyst extracting structured "Customer Truth" from unstructured data. [cite: 1, 14]

### INCLUSION CRITERIA:
- INCLUDE: Mentions of insurer/bank/agent AND contains a real experience, confusion, regret, or detailed account.
- EXCLUDE: Generic praise/criticism, promotional content, or short comments without context.

### TAXONOMY RULES (Choose EXACTLY one per field):
1. Stage: Pre-Sale Misrepresentation, Onboarding & Policy Understanding, Servicing & Policy Admin, Early Regret / Exit Intent, Claims Experience.
2. Root Cause: Sales Pressure or Coercion, Misrepresentation / Incomplete Explanation, Product Design or Complexity, Process / Documentation Friction, Communication / Language Barrier.
3. Channel: Bank / Bancassurance, Agent / Advisor, Branch / Customer Service, Digital / App / Website, Claims / TPA, Unknown.
4. Emotion: Anger, Betrayal, Confusion, Resignation.
5. Severity: High, Medium, Low.

### OUTPUT SCHEMA:
Return a JSON array of objects. Each object must have:
{
    "include": true/false,
    "product_hint": "life/health/savings/unknown",
    "stage": "current stage of complaint",
    "primary_breakdown": "where trust FIRST broke (may differ from stage)",
    "root_cause": "allowed value",
    "channel": "allowed value",
    "emotion": "allowed value",
    "severity": "allowed value",
    "summary": "brief English summary of the 'truth' found"
}

COMMENTS:
"""

# =========================
# 📂 LOAD DATA
# =========================
def load_all_yt_data(directory):
    all_data = []
    json_paths = glob.glob(os.path.join(directory, "**/*.info.json"), recursive=True)

    for j_path in json_paths:
        try:
            with open(j_path, 'r', encoding='utf-8') as f:
                meta = json.load(f)
            all_data.append({
                'title': meta.get('title'),
                'video_id': meta.get('id'),
                'comments': meta.get('comments', [])
            })
        except Exception as e:
            print(f"Error loading {j_path}: {e}")
    return all_data

# =========================
# 🔍 FILTER + DEDUPLICATE
# =========================
def is_fuzzy_match(word, target, threshold=0.8):
    return SequenceMatcher(None, word.lower(), target.lower()).ratio() >= threshold

def contains_icici_lombard_fuzzy(text):
    words = text.lower().split()
    # Check for ICICI or Lombard to capture broader mentions
    return any(is_fuzzy_match(w, "icici") or is_fuzzy_match(w, "lombard") for w in words)

def normalize_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# =========================
# 🕒 TIME + LINK
# =========================
def convert_timestamp_ist(ts):
    if ts:
        return (datetime.utcfromtimestamp(ts) + timedelta(hours=5, minutes=30)).strftime("%Y-%m-%d %H:%M:%S")
    return None

def generate_comment_link(video_id, comment_id):
    if video_id and comment_id:
        return f"https://www.youtube.com/watch?v={video_id}&lc={comment_id}"
    return None

# =========================
# 🤖 BATCH GEMINI CALL
# =========================
def process_batch(comments, retries=3):
    for attempt in range(retries):
        try:
            joined = "\n\n---\n\n".join(comments)
            response = client.models.generate_content(
                model="gemini-2.5-flash-lite", # Using latest stable lite model
                contents=PREPROMPT + joined,
                config={
                    "temperature": 0.1, # Lower temperature for classification accuracy 
                    "response_mime_type": "application/json"
                }
            )
            return json.loads(response.text.strip())
        except Exception as e:
            print(f"Gemini Error (Attempt {attempt+1}): {e}")
            time.sleep(10)
    return None

# =========================
# 🚀 MAIN PIPELINE
# =========================
base_dir = "ICICI_Lombard_Today"
raw_dataset = load_all_yt_data(base_dir)

# 1. Filter: Enforcing length and brand mention 
filtered_comments = [
    (video, comment) 
    for video in raw_dataset 
    for comment in video['comments']
    if (len(comment.get('text', '')) > 200 and contains_icici_lombard_fuzzy(comment.get('text', ''))) # 200 length and words match
]

# 2. Deduplicate: High quality bar requirement 
deduped_comments = []
seen_norms = []
for video, comment in filtered_comments:
    norm = normalize_text(comment.get('text', ''))
    if not any(SequenceMatcher(None, norm, existing).ratio() > 0.85 for existing in seen_norms):
        seen_norms.append(norm)
        deduped_comments.append((video, comment))

print(f"Starting analysis on {len(deduped_comments)} unique comments...")

# 3. Batch processing
final_dataset = []
for i in range(0, len(deduped_comments), BATCH_SIZE):
    batch = deduped_comments[i:i+BATCH_SIZE]
    texts = [c[1].get('text', '') for c in batch]
    results = process_batch(texts)

    if not results or len(results) != len(batch):
        continue

    for (video, comment), result in zip(batch, results):
        if result.get("include"):
            final_dataset.append({
                "insurer": "ICICI Lombard",
                "country": "India",
                "source": "YouTube",
                "link": generate_comment_link(video.get("video_id"), comment.get("id")),
                "date_time": convert_timestamp_ist(comment.get("timestamp")),
                "original_text": comment.get('text', ''),
                "product_hint": result.get("product_hint"),
                "stage": result.get("stage"),
                "primary_breakdown_point": result.get("primary_breakdown"), # Critical Scope Field 
                "channel": result.get("channel"),
                "root_cause": result.get("root_cause"),
                "emotion": result.get("emotion"),
                "emotion_severity": result.get("severity"),
                "summary_insight": result.get("summary")
            })
    
    print(f"Processed batch {i//BATCH_SIZE + 1}. Current Total: {len(final_dataset)}")
    time.sleep(4) # Rate limit management

# =========================
# 📤 SAVE (D1 Deliverable) [cite: 32]
# =========================
df = pd.DataFrame(final_dataset)
df.to_csv("structured_customer_truth.csv", index=False)
print(f"\n✅ Pipeline Complete. Saved {len(df)} high-quality entries.")

In [2]:
# helpful code to test and see data, NOT MAIN PIPELINE

import os
import json
import glob
import webvtt # install via: pip install webvtt-py

def load_all_yt_data(directory):
    all_data = []
    # Find all .info.json files recursively
    json_paths = glob.glob(os.path.join(directory, "**/*.info.json"), recursive=True)
    
    print(f"Found {len(json_paths)} metadata files. Starting load...")

    for j_path in json_paths:
        with open(j_path, 'r', encoding='utf-8') as f:
            meta = json.load(f)
        
        # Get the base filename to find the matching .vtt
        # e.g., "path/to/file.info.json" -> "path/to/file"
        base_path = j_path.replace('.info.json', '')
        
        # Search for any .vtt file starting with that base name
        vtt_search = glob.glob(f"{base_path}*.vtt")
        
        transcript_text = ""
        if vtt_search:
            try:
                # We take the first matching vtt found
                vtt_file = vtt_search[0]
                transcript_text = " ".join([caption.text for caption in webvtt.read(vtt_file)])
            except Exception as e:
                print(f"Error reading VTT for {meta.get('title')}: {e}")

        # Combine into a single record
        video_record = {
            'title': meta.get('title'),
            'uploader_id': meta.get('uploader_id'),
            'uploader': meta.get('uploader'),
            'upload_date': meta.get('upload_date'),
            'description': meta.get('description'),
            'transcript': transcript_text,
            'comments': meta.get('comments', []),
            'view_count': meta.get('view_count'),
            'comment_count': meta.get('comment_count'),
            'like_count': meta.get('like_count')
        }
        all_data.append(video_record)

    return all_data

# Execute
base_dir = "ICICI_Lombard_Today"
raw_dataset = load_all_yt_data(base_dir)

print(f"Successfully loaded {len(raw_dataset)} total records.")
# Calculate total comments
total_comments = sum(len(video.get('comments', [])) for video in raw_dataset)

print(f"Total comments across all videos: {total_comments}")

Found 547 metadata files. Starting load...
Successfully loaded 547 total records.
Total comments across all videos: 42552


In [3]:
# de duplicate and print data

from difflib import SequenceMatcher
import re

def is_fuzzy_match(word, target, threshold=0.8):
    return SequenceMatcher(None, word, target).ratio() >= threshold

def contains_icici_lombard_fuzzy(text):
    words = text.lower().split()
    return any(
        is_fuzzy_match(w, "icici") or is_fuzzy_match(w, "lombard")
        for w in words
    )

def normalize_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# 1. Filter
filtered_comments = [
    comment for video in raw_dataset 
    for comment in video['comments'] 
    if (
        len(comment.get('text', '')) > 200 and
        contains_icici_lombard_fuzzy(comment.get('text', ''))
    )
]

# 2. Fuzzy Dedup (near-duplicates removed)
deduped_comments = []
seen_norms = []

for comment in filtered_comments:
    text = comment.get('text', '')
    norm = normalize_text(text)

    # Check similarity with already kept comments
    is_duplicate = any(
        SequenceMatcher(None, norm, existing).ratio() > 0.9
        for existing in seen_norms
    )

    if not is_duplicate:
        seen_norms.append(norm)
        deduped_comments.append(comment)

# 3. Sort (Shortest → Longest)
sorted_comments = sorted(
    deduped_comments,
    key=lambda x: len(x.get('text', ''))
)


# 4. Output
print(f"Final comments after fuzzy dedup: {len(sorted_comments)}")
print("-" * 40)

for i, comment in enumerate(sorted_comments):
    text = comment.get('text', '')
    print(f"Comment #{i+1} | Length: {len(text)}")
    print(f"Content: {text[:200]}...")
    print("-" * 20)

Final comments after fuzzy dedup: 417
----------------------------------------
Comment #1 | Length: 201
Content: Mera star comprehensive plan hai...kya mujhe icici Lombard mein switch karna chaiye...koi bol raha hai star  claim reject kar deti hai koi bol raha hai icici Lombard mat lena...kya karoon confusion ha...
--------------------
Comment #2 | Length: 201
Content: Battery Short Circuit ho sakta hai 
Kyuki tu bol rha hai tha start kiya tha car par start hui nahi 

But still ICICI ko IDV jitna compensation dena chahiye 

Best rahega ki tu case karva de inke upar ...
--------------------
Comment #3 | Length: 201
Content: ICICI elevate plan has feature of upgrading single Room to Any room 
so my question is 
if we upgrade to   Any room option so  proportionate  deduction based on  bill amount still applicable or not ...
--------------------
Comment #4 | Length: 201
Content: no sir apko ek rupee bhi bharne padega its is my point of view is your r really knoe about icici lambard insuran

In [1]:
# BELOW IS A SHELL COMMAND NOT PYTHON

# data scraper
# yt-dlp "ytsearchall:icici lombard good" \
#     --download-archive "processed_videos.txt" \
#     --no-overwrites \
#     --write-info-json --write-subs --write-auto-subs --get-comments --skip-download \
#     --extractor-args "youtube:player-client=android,web" \
#     --sleep-requests 0 --sleep-interval 0 \
#     -o "ICICI_Lombard_Today/%(upload_date)s_%(title)s.%(ext)s"

In [ ]:
# to rebuild processed_videos.txt if not working(data scraper skips files in this text file that are already downloaded)
import os
import json
import glob

def rebuild_yt_dlp_archive(directory, archive_filename="processed_videos.txt"):
    # Find all .info.json files in the directory and subdirectories
    json_paths = glob.glob(os.path.join(directory, "**/*.info.json"), recursive=True)
    
    if not json_paths:
        print(f"No .info.json files found in '{directory}'. Check your path.")
        return

    video_ids = set()

    for j_path in json_paths:
        try:
            with open(j_path, 'r', encoding='utf-8') as f:
                meta = json.load(f)
                video_id = meta.get('id')
                # We use a set to ensure no duplicates if files were moved
                if video_id:
                    video_ids.add(video_id)
        except Exception as e:
            print(f"Could not read {j_path}: {e}")

    # Write to the archive file in the format: youtube [ID]
    with open(archive_filename, 'w', encoding='utf-8') as f:
        for vid in sorted(video_ids):
            f.write(f"youtube {vid}\n")

    print(f"Successfully created '{archive_filename}' with {len(video_ids)} entries.")

# Run the script
if __name__ == "__main__":
    base_dir = "ICICI_Lombard_Today" # Change this to your folder name
    rebuild_yt_dlp_archive(base_dir)